Modern Transformer block (Llama-style)

Pre-RMSNorm with SwiGLU FFN, the exact pattern used by Llama 4, Mistral, and DeepSeek

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# RMSNorm is simpler and faster than layerNorm
# used by llama 4, deepseek v3, mistral, gemma, qwen

class RMSNorm(nn.Module):
    # RMSNorm: scale by root mean square, no mean subtraction

    def __init__(self, dims, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dims))

    def forward(self, x):
        # rsqrt = 1 / sqrt(faster than sqrt + divide)
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdims=True) + self.eps)
        return norm * self.weight
# SwiGLU FFN - Gated Linear Unit with Swish activation
# Stores 2/3 of all model parameters (knowledge bank)

class SwiGLU(nn.Module):
    # SwiGLU(x) = W_down(SiLU(W_gate @ x) * W_up @ x)
    def __init__(self, hidden_dim, intermediate_dim):
        super().__init__()
        self.w_gate = nn.Linear(hidden_dim, intermediate_dim, bias = False)
        self.w_up = nn.Linear(hidden_dim, intermediate_dim, bias = False)
        self.w_down = nn.Linear(intermediate_dim, hidden_dim, bias = False)

    def forward(self, x):
        gate = F.silu(self.w_gate(x)) # whih knowledge paths
        up = self.w_gate(x) # raw values
        return self.w_down(gate * up) # gated output

# Modern transformer block (Pre - RMSNorm + residual)
# x = x + Attn(RMSNorm(x); x = x + FFN(RMSNorm(x)))

class ModernTransformerBlock(nn.Module):
    # llama stle decoder block with pre norm and swiglu
    def __init__(self, hidden_dim=4096, num_heads=32, intermediate_dim=14336):
        super().__init__()
        self.norm1 = RMSNorm(hidden_dim)
        self.attn = nn.MultiheadAttention(
            hidden_dim, num_heads, batch_first=True
        )
        self.norm2 = RMSNorm(hidden_dim)
        self.ffn = SwiGLU(hidden_dim, intermediate_dim)

    def forward(self, x, mask=None):
        # pre norm + attn + residual 
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=mask)
        x = x + attn_out
        # Pre-norm + SwiGLU FFN + residual
        x = x + self.ffn(self.norm2(x))
        return x

block = ModernTransformerBlock(hidden_dim = 512, num_heads=8, intermediate_dim=2048)
x = torch.randn(2, 10, 512)
output = block(x)

total = sum(p.numel() for p in block.parameters())
attn_p = sum(p.numel() for p in block.attn.parameters())
ffn_p = sum(p.numel() for p in block.ffn.parameters())

print(f"Input:  {x.shape} -> Output: {output.shape}")
print(f"Total params:  {total:,}")
print(f"  Attention:   {attn_p:,} ({100*attn_p//total}%)")
print(f"  FFN (SwiGLU):{ffn_p:,} ({100*ffn_p//total}%)")

Input:  torch.Size([2, 10, 512]) -> Output: torch.Size([2, 10, 512])
Total params:  4,197,376
  Attention:   1,050,624 (25%)
  FFN (SwiGLU):3,145,728 (74%)


Simplified MoE layer

How MoE works: N expert FFNs, only top-K activate per token (simplified top-2 example)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# mixture of experts
# idea: n experts ffns but keep only top k activate per token
# total params = n * experts params (knowledge capacity)
# active params = k * experts params (compute cost)

class SwiGLUExpert(nn.Module):
    # each expert is a full SwiGLU FFN.
    def __init__(self, hidden_dim, intermediate_dim):
        super().__init__()
        self.w_gate = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.w_up = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.w_down = nn.Linear(intermediate_dim, hidden_dim, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class SimpleMoE(nn.Module):
    # simplified moe layer : n experts, top k routing per token

    def __init__(self, hidden_dim, intermediate_dim, num_experts = 8, top_k = 2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(hidden_dim, num_experts, bias= False)
        self.experts = nn.ModuleList([
            SwiGLUExpert(hidden_dim, intermediate_dim) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, S, D = x.shape
        x_flat = x.view(-1, D)

        # route which expert for each token
        logits = self.router(x_flat)
        weights, indices = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(weights, dim=-1)
    
        # compute weighted expert outputs
        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for i, expert in enumerate(self.experts):
                mask = (indices[:, k] == i)
                if mask.any():
                    out = expert(x_flat[mask])
                    output[mask] += weights [mask, k:k+1] * out
        
        return output.view(B, S, D)

# Example: 8 experts, top-2 routing
moe = SimpleMoE(hidden_dim=512, intermediate_dim=1024,
                num_experts=8, top_k=2)
x = torch.randn(2, 10, 512)
output = moe(x)

total = sum(p.numel() for p in moe.parameters())
expert_p = sum(p.numel() for p in moe.experts[0].parameters())
active = 2 * expert_p  # top-2

print(f"Output: {output.shape}")
print(f"Total params:  {total:,}")
print(f"Active params: {active:,} ({100*active/total:.1f}%)")
print(f"\nDeepSeek V3 equivalent:")
print(f"  256 experts, top-8 routing")
print(f"  671B total -> ~37B active per token (5.5%)")

Output: torch.Size([2, 10, 512])
Total params:  12,587,008
Active params: 3,145,728 (25.0%)

DeepSeek V3 equivalent:
  256 experts, top-8 routing
  671B total -> ~37B active per token (5.5%)
